# UK daily basket: build a 30-day price history

Reads the free daily basket dataset `ynAT9NPps2EdjMOJa`, published by
[UK Grocery Prices](https://apify.com/yappman/uk-supermarket-price-scraper).
It holds the same 212 staple products from four retailers, priced every day
since 24 September 2026. The notebook builds a per-product price history for
the last 30 days and a simple chained index for each retailer.

**Cost at Apify Free-tier prices: $0.00.** Reading a dataset starts no Actor
and charges no events. There is nothing to cap, so this example has no
`maxItems`. The history window is the cap: reading stops at the first row
older than 30 days.

The dataset has had fewer than 30 days of history until late October 2026.
The notebook uses whatever days exist and says how many it found.

More history and the weekly catalogue: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from apify_client import ApifyClient

# The dataset is public, so the token is optional here; it is still read from
# the environment, never from the notebook.
client = ApifyClient(os.environ.get("APIFY_TOKEN"))

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)  # git-ignored

DATASET_ID = "ynAT9NPps2EdjMOJa"
DAYS = 30

## 1. Read the last 30 days, newest first

Rows are appended once a day, so reading in descending order and stopping at
the cutoff reads only the window needed.

In [ ]:
FIELDS = ["retailer", "retailerProductId", "name", "packSize",
          "price", "loyaltyPrice", "inStock", "scrapedAt"]
cutoff = pd.Timestamp.now(tz="UTC").normalize() - pd.Timedelta(days=DAYS - 1)

rows = []
for item in client.dataset(DATASET_ID).iterate_items(fields=FIELDS, desc=True):
    if pd.Timestamp(item["scrapedAt"]) < cutoff:
        break
    rows.append(item)

df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["scrapedAt"], utc=True).dt.tz_localize(None).dt.normalize()
df["productKey"] = df["retailer"] + ":" + df["retailerProductId"].astype(str)
df = (df.sort_values("scrapedAt")
        .drop_duplicates(["productKey", "date"], keep="last")
        .dropna(subset=["price"]))
df["effectivePrice"] = df[["price", "loyaltyPrice"]].min(axis=1, skipna=True)

n_days = df["date"].nunique()
print(f"{len(df)} rows, {df['productKey'].nunique()} products, {n_days} days "
      f"({df['date'].min():%d %b} to {df['date'].max():%d %b %Y})")
if n_days < DAYS:
    print(f"Only {n_days} of {DAYS} days exist yet; the history uses all of them.")
df.groupby("retailer")["date"].nunique().rename("days")

## 2. Price history per product

In [ ]:
history = df.pivot_table(index="date", columns="productKey", values="effectivePrice")
names = df.drop_duplicates("productKey").set_index("productKey")[["retailer", "name", "packSize"]]

stats = names.join(pd.DataFrame({
    "first": history.bfill().iloc[0],
    "latest": history.ffill().iloc[-1],
    "low": history.min(),
    "high": history.max(),
    "priceChanges": history.ffill().diff().fillna(0).ne(0).sum(),
}))
stats["changePct"] = ((stats["latest"] / stats["first"] - 1) * 100).round(1)

history.to_csv(OUTPUT / "basket_history_wide.csv")
stats.to_csv(OUTPUT / "basket_history_summary.csv")
stats.sort_values("changePct").head(10)

In [ ]:
stats.sort_values("changePct", ascending=False).head(10)

## 3. A chained index per retailer

Each day's index moves by the geometric mean of price relatives for products
priced on both that day and the day before (a chained Jevons index). The first
day is 100. This is a quick illustration, not the published
[Shelf-Price Index](https://data.yappman.com/shelf-price-index).

In [ ]:
def chained_jevons(prices: pd.DataFrame) -> pd.Series:
    rel = prices / prices.shift(1)
    links = np.exp(np.log(rel).mean(axis=1, skipna=True)).fillna(1.0)
    links.iloc[0] = 1.0
    return 100 * links.cumprod()


index = pd.DataFrame({
    retailer: chained_jevons(history[[k for k in history.columns if k.startswith(retailer + ":")]])
    for retailer in sorted(df["retailer"].unique())
})

ax = index.plot(figsize=(9, 4), marker="o", ms=3)
ax.set_title("Daily basket, chained index (first day = 100)")
ax.set_ylabel("Index")
ax.set_xlabel("")
ax.grid(alpha=0.3)
plt.tight_layout()
index.round(2).tail()

## Notes

* Effective price is the lower of shelf and loyalty-card price.
* The basket rows are the Actor publisher's data. A free sample is on
  [Kaggle](https://www.kaggle.com/datasets/russellyapp/uk-supermarket-prices-daily-212-item-basket)
  under CC BY 4.0. Nothing collected here is committed: `output/` is git-ignored.